In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np

from utils.phase import Phase
from utils.model_utils import criterion_to_str
from utils.model_utils import compare_results_strata
from data_augmentation.monte_carlo_loader import MonteCarloLoader
from data_augmentation.datasets.dense_monte_carlo_dataset import DenseMonteCarloDataset
from model.architecture.dense_monte_carlo import DenseMonteCarloNet
from model.architecture.residual_connection_dense import ResDeepAHPNet
from model.loss_function.dense_monte_carlo.custom_dense_MC_loss import CustomDenseMCLoss
from model.model_loader import ModelLoader
from model.trainers.dense_monte_carlo_trainer import DenseMonteCarloTrainer
from analytic_methods.EVM_method import EVM
from visualization.results.metrics_visualizator import MetricsVisualizator
from visualization.data.weights_visualizator import WeightsVisualizator
from metrics.evaluator import Evaluator

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device used: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
CRITERIA_NUM = 5
CRITERIA_NUMS = [3, 4, 5, 6, 7, 8, 9]
STRATA = ['low_inconsistency', 'moderate_inconsistency', 'high_inconsistency']
NORMALIZE_INITIAL_LOSS = True # dopisać o tym w pracy 
TRAIN_MATRICES_NUM = None
VALID_MATRICES_NUM = None
TEST_MATRICES_NUM = None

BATCH_SIZE = 1024
LEARNING_RATE = 5e-4
COP = 0.45
FIT = 0.45
STAB = 0.10
P = 1 # 1 = MAE, 2 = MSE
MARGIN = 0.05
EPOCHS = 500
COP_MODE = "margin" 
STAB_MODE = "kullback"
    # COP, FIT, STAB, P, MARGIN, NORMALIZE
PARAMETERS = [
    [0.5, 0.45, 0.05, 1, 0.05, True],
    [0.5, 0.45, 0.05, 1, 0.05, False],
    [0.5, 0.45, 0.05, 2, 0.05, False],
    [0.7, 0.25, 0.05, 1, 0.05, True],
]

if 'trainer' not in locals():
    criterion_str = criterion_to_str(COP, FIT, STAB, MARGIN, P, COP_MODE, STAB_MODE, NORMALIZE_INITIAL_LOSS)
    model_loader = ModelLoader()
    model, history = model_loader.load_dense_MC(CRITERIA_NUM, criterion_str)
    criterion = CustomDenseMCLoss(n_criteria=CRITERIA_NUM, lambda_cop=COP, lambda_fit=FIT, lambda_stab=STAB, p=P, 
        normalize_initial_loss=NORMALIZE_INITIAL_LOSS, margin=MARGIN).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    trainer = DenseMonteCarloTrainer(model, optimizer, criterion, device)

In [ ]:
loader = MonteCarloLoader()
max_test_per_stratum = TEST_MATRICES_NUM // len(STRATA) if TEST_MATRICES_NUM else None

matrices = {}
weights = {}
predicted_priority_weights = {}
evm_solver = EVM()
evm_priority_weights = {}

for stratum in STRATA:
    test_mats, test_wts = loader.load_stratum(
        phase=Phase.TEST,
        num_criteria=CRITERIA_NUM,
        stratum=stratum,
        max_count=max_test_per_stratum
    )
    matrices[stratum] = test_mats
    weights[stratum] = test_wts
    predicted_priority_weights[stratum] = trainer.predict_batch(test_mats)
    evm_pw, _ = evm_solver.compute_batch(test_mats)
    evm_priority_weights[stratum] = evm_pw

print("Predicted AHP weights for all strata.")

In [ ]:
weights_visualizator = WeightsVisualizator()
nn_name = f"Dense Monte Carlo (N={CRITERIA_NUM})"

for stratum_key in weights.keys():
    pred_weights_vector = predicted_priority_weights[stratum_key][0]
    evm_weights_vector = evm_priority_weights[stratum_key][0].flatten()
    weights_vector = weights[stratum_key][0].flatten()
    weights_visualizator.plot_comparison_all(weights_vector, pred_weights_vector, evm_weights_vector, consistency_level=str(stratum_key).split("_")[0], nn_name=nn_name)

In [ ]:
evm_solver = EVM()
evaluator = Evaluator()
nn_name = f"Dense Monte Carlo (N={CRITERIA_NUM})"

compare_results_strata(evm_solver, evaluator, matrices, weights, predicted_priority_weights, nn_name)

In [ ]:
viz = MetricsVisualizator()
viz.display_loss(history, "Dense Monte Carlo")
viz.display_part_losses(history, "Dense Monte Carlo")